# CS156 Professor Live Translator Tester

This notebook is the clean live-testing version of the project. It does **not** use Vercel, Cloudflare tunnels, environment variables, or any web app backend.

Run it in Google Colab with a GPU runtime. The UI lets a professor type English text and compare:

- RNN LSTM from scratch
- Transformer from scratch
- OPUS-MT pretrained
- OPUS-MT fine-tuned
- Meta NLLB-200 distilled 600M checker

Keep the project folder in Google Drive at `MyDrive/CS156 Final Assignment`, or edit `PROJECT_DIR` in the setup cell.


In [ ]:
# Setup: install dependencies, mount Drive, and locate the project folder.
%pip -q install sacrebleu transformers sentencepiece sacremoses accelerate ipywidgets

from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Google Drive mount skipped or failed: {exc}')

PROJECT_DIR_CANDIDATES = [
    Path('/content/drive/MyDrive/CS156 Final Assignment'),
    Path('/content/drive/MyDrive/Downloads/CS156 Final Assignment'),
    Path.cwd(),
]

PROJECT_DIR = None
for candidate in PROJECT_DIR_CANDIDATES:
    if candidate.exists() and (
        (candidate / 'RNN Outputs' / 'rnn_best.pt').exists()
        or (candidate / 'OPUS_MT Outputs').exists()
        or (candidate / 'final_model_evaluation.py').exists()
    ):
        PROJECT_DIR = candidate.resolve()
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find the project folder. Put the project at /content/drive/MyDrive/CS156 Final Assignment '
        'or edit PROJECT_DIR_CANDIDATES in this cell.'
    )

os.chdir(PROJECT_DIR)
print('Using project folder:', PROJECT_DIR)

required_paths = [
    'final_model_evaluation.py',
    'RNN Outputs/rnn_best.pt',
    'OPUS_MT Outputs/1_direct_pretrained_model',
    'OPUS_MT Outputs/2_fine_tuned_model',
]
transformer_ok = any((PROJECT_DIR / folder).exists() for folder in ['Transformers Outputs', 'Transformer Outputs'])

print('\nProject file check:')
for item in required_paths:
    print(('OK   ' if (PROJECT_DIR / item).exists() else 'MISS ') + item)
print(('OK   ' if transformer_ok else 'MISS ') + 'Transformers Outputs or Transformer Outputs')


In [ ]:
# Translation engine. This reuses final_model_evaluation.py for the four project models.
from __future__ import annotations

import gc
import importlib
import shutil
import tempfile
import time
from pathlib import Path

import pandas as pd

import final_model_evaluation as fme
importlib.reload(fme)

# Make each UI run generate a fresh one-row prediction instead of reading cached CSVs.
fme.FORCE_REGENERATE = True
fme.GENERATE_MISSING_PREDICTIONS = True
fme.USE_EXISTING_OPUS_PREVIEWS = False

MODEL_LABELS = {
    'rnn_lstm_scratch': 'RNN LSTM from scratch',
    'transformer_scratch': 'Transformer from scratch',
    'opus_mt_pretrained': 'OPUS-MT pretrained',
    'opus_mt_fine_tuned': 'OPUS-MT fine-tuned',
    'nllb_checker': 'Meta NLLB-200 distilled checker',
}

PROJECT_DIR = Path(PROJECT_DIR)


def make_live_dirs():
    temp_base = Path('/content') if Path('/content').exists() else Path(tempfile.gettempdir())
    root = Path(tempfile.mkdtemp(prefix='cs156_live_', dir=str(temp_base)))
    dirs = {
        'root': root,
        'predictions': root / 'predictions',
        'tables': root / 'tables',
        'figures': root / 'figures',
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs, root


def one_row_dataframe(text: str) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                'row_id': 0,
                'source': 'Live professor test',
                'English': text,
                'Reference Igbo': '',
            }
        ]
    )


def run_project_model(text: str, generator):
    dirs, root = make_live_dirs()
    try:
        df = one_row_dataframe(text)
        pred_df = generator(PROJECT_DIR, dirs, df)
        return str(pred_df.loc[0, 'Predicted Igbo'])
    finally:
        shutil.rmtree(root, ignore_errors=True)
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass


def translate_rnn(text: str) -> str:
    return run_project_model(text, fme.generate_rnn_predictions)


def translate_transformer(text: str) -> str:
    output = run_project_model(text, fme.generate_transformer_predictions)
    try:
        import tensorflow as tf
        tf.keras.backend.clear_session()
    except Exception:
        pass
    return output


def translate_opus_pretrained(text: str) -> str:
    return run_project_model(
        text,
        lambda project_dir, dirs, df: fme.generate_opus_predictions(
            project_dir,
            dirs,
            df,
            model_slug='opus_mt_pretrained',
            model_dir_name='1_direct_pretrained_model',
        ),
    )


def translate_opus_fine_tuned(text: str) -> str:
    return run_project_model(
        text,
        lambda project_dir, dirs, df: fme.generate_opus_predictions(
            project_dir,
            dirs,
            df,
            model_slug='opus_mt_fine_tuned',
            model_dir_name='2_fine_tuned_model',
        ),
    )


_NLLB_TOKENIZER = None
_NLLB_MODEL = None
_NLLB_DEVICE = None
_NLLB_FORCED_BOS_TOKEN_ID = None


def translate_nllb(text: str) -> str:
    global _NLLB_TOKENIZER, _NLLB_MODEL, _NLLB_DEVICE, _NLLB_FORCED_BOS_TOKEN_ID
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    model_name = 'facebook/nllb-200-distilled-600M'
    if _NLLB_MODEL is None:
        print('Loading NLLB checker. This is the largest model and may take several minutes the first time.')
        _NLLB_TOKENIZER = AutoTokenizer.from_pretrained(model_name, src_lang='eng_Latn')
        model_kwargs = {}
        if torch.cuda.is_available():
            model_kwargs['torch_dtype'] = torch.float16
        _NLLB_MODEL = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
        _NLLB_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        _NLLB_MODEL.to(_NLLB_DEVICE)
        _NLLB_MODEL.eval()
        _NLLB_FORCED_BOS_TOKEN_ID = _NLLB_TOKENIZER.convert_tokens_to_ids('ibo_Latn')

    encoded = _NLLB_TOKENIZER([text], return_tensors='pt', padding=True, truncation=True, max_length=200)
    encoded = {key: value.to(_NLLB_DEVICE) for key, value in encoded.items()}
    with torch.no_grad():
        generated = _NLLB_MODEL.generate(
            **encoded,
            forced_bos_token_id=_NLLB_FORCED_BOS_TOKEN_ID,
            max_new_tokens=100,
            num_beams=2,
        )
    return _NLLB_TOKENIZER.batch_decode(generated, skip_special_tokens=True)[0]


def clear_loaded_checker():
    global _NLLB_TOKENIZER, _NLLB_MODEL, _NLLB_DEVICE, _NLLB_FORCED_BOS_TOKEN_ID
    _NLLB_TOKENIZER = None
    _NLLB_MODEL = None
    _NLLB_DEVICE = None
    _NLLB_FORCED_BOS_TOKEN_ID = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


TRANSLATORS = {
    'rnn_lstm_scratch': translate_rnn,
    'transformer_scratch': translate_transformer,
    'opus_mt_pretrained': translate_opus_pretrained,
    'opus_mt_fine_tuned': translate_opus_fine_tuned,
    'nllb_checker': translate_nllb,
}

print('Translation engine ready.')


In [ ]:
# Professor-facing UI.
import html
import time

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display

english_input = widgets.Textarea(
    value='How are you?',
    placeholder='Type an English sentence here...',
    description='English',
    layout=widgets.Layout(width='100%', height='110px'),
)

model_checks = {
    key: widgets.Checkbox(value=True, description=label, indent=False)
    for key, label in MODEL_LABELS.items()
}

run_button = widgets.Button(
    description='Translate',
    button_style='primary',
    tooltip='Run the selected models on the English text',
)
clear_button = widgets.Button(
    description='Clear NLLB cache',
    button_style='',
    tooltip='Free GPU memory used by the NLLB checker',
)

status_out = widgets.Output()
results_out = widgets.Output()


def selected_models():
    return [key for key, checkbox in model_checks.items() if checkbox.value]


def show_results(rows):
    with results_out:
        clear_output(wait=True)
        if not rows:
            return
        df = pd.DataFrame(rows)
        display(HTML(df.to_html(index=False, escape=True).replace('\n', '<br>')))


def on_translate(_):
    text = english_input.value.strip()
    if not text:
        with status_out:
            clear_output(wait=True)
            print('Type an English sentence first.')
        return

    models = selected_models()
    if not models:
        with status_out:
            clear_output(wait=True)
            print('Select at least one model.')
        return

    rows = []
    run_button.disabled = True
    with status_out:
        clear_output(wait=True)
        print('Running selected models. The first run can take several minutes, especially NLLB.')

    for key in models:
        label = MODEL_LABELS[key]
        started = time.time()
        with status_out:
            print(f'[{time.strftime("%H:%M:%S")}] Running {label}...')
        try:
            prediction = TRANSLATORS[key](text)
            status = 'ok'
        except Exception as exc:
            prediction = f'{type(exc).__name__}: {exc}'
            status = 'error'
        elapsed = round(time.time() - started, 1)
        rows.append(
            {
                'Model': label,
                'Status': status,
                'Seconds': elapsed,
                'Igbo output': prediction,
            }
        )
        show_results(rows)

    with status_out:
        print(f'[{time.strftime("%H:%M:%S")}] Done.')
    run_button.disabled = False


def on_clear(_):
    clear_loaded_checker()
    with status_out:
        print('Cleared NLLB checker from memory.')


run_button.on_click(on_translate)
clear_button.on_click(on_clear)

controls = widgets.VBox([
    HTML('<h2>CS156 English to Igbo Live Translator Tester</h2>'),
    english_input,
    widgets.HTML('<b>Select models to run:</b>'),
    widgets.VBox(list(model_checks.values())),
    widgets.HBox([run_button, clear_button]),
    widgets.HTML('<p><b>Note:</b> first run is slow because models must load from Drive or Hugging Face.</p>'),
])

display(controls)
display(status_out)
display(results_out)
